# v0.3.6: 100-Neuron Excitatory-Inhibitory Population

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HNXJ/jaxfne/blob/main/tutorials/jaxfne_v036_100_neuron_ei_population.ipynb)

**Duration:** 15–20 minutes | **Difficulty:** Intermediate | **v0.3.5+**

## 1. Learning Objectives

By the end of this tutorial, you will:

- Build a balanced excitatory-inhibitory (E/I) population using the chainable Configuration API
- Simulate 100 neurons (75 E, 25 I) with Izhikevich dynamics and all-to-all connectivity
- Extract multimodal readouts: spike rasters, voltage, source density, and field proxies
- Generate population-level metrics (firing rates, voltage statistics, source power)
- Inspect manifest and scope metadata for computational validation
- Visualize population dynamics and field patterns

## 2. Biological/Computational Question

**Question:** How do excitatory and inhibitory neurons interact in a balanced population, and what aggregate dynamics emerge from their coupling?

**Context:** 
Cortical populations are typically composed of ~80% excitatory (E) pyramidal neurons and ~20% inhibitory (I) interneurons. This 4:1 ratio produces balanced network dynamics where excitation and inhibition roughly match in strength—a key principle in neuroscience.

**Approach:**
We configure a 100-neuron population (75 E + 25 I) with all-to-all coupling. E neurons excite both E and I targets via AMPA receptors. I neurons inhibit both E and I targets via GABA receptors. We simulate 1 second of activity and measure population rates, voltage distributions, and field signatures.

**Scope:** This is a *computational scaffold* for learning E/I coupling. The Izhikevich model is phenomenological, not biophysically detailed. Field readouts are proxy approximations suitable for visualization and relative comparison.

## 3. Mathematical Glossary Flow

### Izhikevich Dynamics

Each neuron n is governed by two state variables:

- **v_n(t)** [mV]: Membrane potential
- **u_n(t)** [µA]: Membrane recovery variable (adaptation)

**Differential equations:**

$$\frac{dv_n}{dt} = 0.04 v_n^2 + 5 v_n + 140 - u_n + I_n(t)$$

$$\frac{du_n}{dt} = a_n (b_n v_n - u_n)$$

**Spike reset:** When $v_n \geq 30$ mV, a spike occurs and states reset:
$$v_n \leftarrow c_n, \quad u_n \leftarrow u_n + d_n$$

**Synaptic input:**
$$I_n(t) = I_{\text{injected}} + \sum_j W_{nj} s_j(t)$$

where $W_{nj}$ is synaptic weight and $s_j(t)$ is postsynaptic conductance (AMPA/GABA).

### Population Metrics

**Mean firing rate [Hz]:**
$$f = \frac{\sum_n \text{spikes}_n}{T \cdot N}$$

**Source density [arbitrary units]:**
$$S(t) = \sum_n I_{\text{mem}, n}(t) \cdot \text{anatomy}(n)$$

**LFP proxy (spatial average):**
$$\phi_{\text{proxy}}(t) = \sum_n W_n S_n(t)$$

**CSD proxy (second spatial derivative):**
$$\text{CSD}(t) \approx \nabla^2 \phi_{\text{proxy}}(t)$$

## 4. Canonical Import

In [ ]:
import jaxfne as jtfne
import jax.numpy as jnp
import numpy as np
import json
import os
from pathlib import Path

print(f"jaxfne version: {jtfne.__version__}")

## 5. Configuration Block

In [ ]:
# Create configuration
cfg = jtfne.Configuration()

# Set runtime parameters
cfg = cfg.runtime(
    seed=42,
    dtype="float32",
    duration_ms=1000.0,
    dt_ms=0.1
)

# Define cortical column: 100 neurons in L2/3
cfg = cfg.column(
    name="L2/3_column",
    layers=["L2/3"],
    n=100
)

# Set E/I composition: 75% E, 25% I
cfg = cfg.cell_types({"E": 0.75, "I": 0.25})

# All-to-all connectivity with standard kinetics
cfg = cfg.connectivity()

# Izhikevich neuron model with cortical parameters
cfg = cfg.set_emitter(
    family="izhikevich",
    preset="cortical_eig"
)

# Multimodal probes: spikes, voltage, source, and field proxies
cfg = cfg.probes([
    "SPK",           # Spike detection
    "Vm",            # Membrane voltage
    "source",        # Transmembrane current → source density
    "LFP-proxy",     # Local field potential (proxy)
    "CSD-proxy"      # Current source density (proxy)
])

print("✓ Configuration complete:")
print(f"  - 100 neurons (75 E / 25 I)")
print(f"  - Izhikevich (cortical_eig preset)")
print(f"  - 1000 ms duration, 0.1 ms timestep")
print(f"  - Deterministic seed: 42")

## 6. Simulation Block

In [ ]:
# Construct model from configuration
model = jtfne.construct(cfg)
print(f"✓ Model constructed")

# Run simulation
signals = jtfne.simulate(
    model,
    duration_ms=1000.0,
    dt_ms=0.1,
    seed=42
)

print(f"✓ Simulation complete")
print(f"  - Timesteps: {signals.V_m.shape[0]}")
print(f"  - Neurons: {signals.V_m.shape[1]}")
print(f"  - Voltage range: [{signals.V_m.min():.1f}, {signals.V_m.max():.1f}] mV")
print(f"  - Total spikes: {signals.spikes.sum():.0f}")

## 7. Probe/Readout Block

In [ ]:
# Compute readout metrics
readout_specs = [
    jtfne.readout_spec("population_rate", "spike_rate_hz"),
    jtfne.readout_spec("mean_voltage", "mean_V_m"),
    jtfne.readout_spec("max_voltage", "max_V_m"),
    jtfne.readout_spec("min_voltage", "min_V_m"),
]

readouts = model.compute_readout(signals, readout_specs)

print("✓ Readout metrics:")
for key, val in readouts.results.items():
    if val is not None:
        print(f"  - {key}: {val:.2f}")

# Separate E and I firing rates
n_E = 75
spikes_E = signals.spikes[:, :n_E]
spikes_I = signals.spikes[:, n_E:]
dt_ms = 0.1
duration_s = 1.0

rate_E = spikes_E.sum() / (n_E * duration_s)  # Hz
rate_I = spikes_I.sum() / (25 * duration_s)   # Hz
rate_pop = (spikes_E.sum() + spikes_I.sum()) / (100 * duration_s)  # Hz

print(f"\n✓ Population rates:")
print(f"  - Excitatory (E): {rate_E:.2f} Hz")
print(f"  - Inhibitory (I): {rate_I:.2f} Hz")
print(f"  - Population mean: {rate_pop:.2f} Hz")
print(f"  - In valid range [2, 25] Hz: {2 <= rate_pop <= 25}")

## 8. Manifest and Scope Metadata

In [ ]:
# Create output directory
output_dir = Path("tutorial_outputs/v036_100_neuron_ei_population")
output_dir.mkdir(parents=True, exist_ok=True)
figures_dir = output_dir / "figures"
figures_dir.mkdir(exist_ok=True)

# Build manifest
manifest = {
    "metadata": {
        "tutorial": "v036_100_neuron_ei_population",
        "jaxfne_version": jtfne.__version__,
        "date_created": str(np.datetime64('today')),
    },
    "configuration": {
        "n_neurons": 100,
        "n_excitatory": 75,
        "n_inhibitory": 25,
        "emitter_family": "izhikevich",
        "emitter_preset": "cortical_eig",
        "connectivity_mode": "all_to_all",
    },
    "simulation": {
        "duration_ms": 1000.0,
        "dt_ms": 0.1,
        "dtype": "float32",
        "seed": 42,
        "timesteps": int(signals.V_m.shape[0]),
    },
    "numerical_results": {
        "excitatory_rate_hz": float(rate_E),
        "inhibitory_rate_hz": float(rate_I),
        "population_mean_rate_hz": float(rate_pop),
        "voltage_min_mv": float(signals.V_m.min()),
        "voltage_max_mv": float(signals.V_m.max()),
        "voltage_mean_mv": float(signals.V_m.mean()),
        "finite_outputs": bool(jnp.isfinite(signals.V_m).all()),
        "finite_spikes": bool(jnp.isfinite(signals.spikes).all()),
    },
    "scope": {
        "scope_status": "computational_scaffold",
        "calibration_status": "uncalibrated_phenomenological",
        "readout_status": "proxy_scale",
        "field_mode": "proxy_convolution_no_pde",
        "proxy_scale": True,
        "physical_amplitude_claim_allowed": False,
    },
    "figures": {
        "01_population_raster": str(figures_dir / "01_population_raster.png"),
        "02_population_rate": str(figures_dir / "02_population_rate.png"),
        "03_voltage_samples": str(figures_dir / "03_voltage_samples.png"),
        "04_source_summary": str(figures_dir / "04_source_summary.png"),
        "05_readout_summary": str(figures_dir / "05_readout_summary.png"),
    }
}

# Save manifest
manifest_path = output_dir / "manifest.json"
with open(manifest_path, 'w') as f:
    json.dump(manifest, f, indent=2)

print(f"✓ Manifest saved: {manifest_path}")
print(f"  - Scope: {manifest['scope']['scope_status']}")
print(f"  - Field mode: {manifest['scope']['field_mode']}")
print(f"  - Physical claims allowed: {manifest['scope']['physical_amplitude_claim_allowed']}")

## 9. Figures

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Figure 1: Population spike raster
fig, ax = plt.subplots(figsize=(12, 6))
spike_times, neuron_ids = jnp.where(signals.spikes > 0.5)
spike_times = spike_times * 0.1  # Convert to ms
ax.scatter(spike_times, neuron_ids, s=1, alpha=0.5, c=['red' if nid < 75 else 'blue' for nid in neuron_ids])
ax.set_xlabel('Time (ms)')
ax.set_ylabel('Neuron ID')
ax.set_title('Population Spike Raster (Red=E, Blue=I)')
red_patch = mpatches.Patch(color='red', label='Excitatory (n=75)')
blue_patch = mpatches.Patch(color='blue', label='Inhibitory (n=25)')
ax.legend(handles=[red_patch, blue_patch])
ax.set_xlim(0, 1000)
ax.set_ylim(-1, 100)
plt.tight_layout()
plt.savefig(figures_dir / "01_population_raster.png", dpi=100, bbox_inches='tight')
plt.close()
print("✓ Figure 1: Population spike raster")

# Figure 2: Firing rate over time
fig, ax = plt.subplots(figsize=(12, 4))
window_size = 100  # 100 timesteps = 10 ms
rate_window_E = []
rate_window_I = []
times_window = []
for i in range(0, len(signals.spikes) - window_size, window_size):
    rate_E_w = signals.spikes[i:i+window_size, :75].sum() / (75 * 0.01)  # Hz
    rate_I_w = signals.spikes[i:i+window_size, 75:].sum() / (25 * 0.01)  # Hz
    rate_window_E.append(rate_E_w)
    rate_window_I.append(rate_I_w)
    times_window.append((i + window_size/2) * 0.1)
ax.plot(times_window, rate_window_E, label='Excitatory', color='red', linewidth=2)
ax.plot(times_window, rate_window_I, label='Inhibitory', color='blue', linewidth=2)
ax.axhline(y=rate_pop, color='black', linestyle='--', label=f'Pop mean: {rate_pop:.1f} Hz')
ax.set_xlabel('Time (ms)')
ax.set_ylabel('Firing Rate (Hz)')
ax.set_title('Population Firing Rate (10 ms window)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(figures_dir / "02_population_rate.png", dpi=100, bbox_inches='tight')
plt.close()
print("✓ Figure 2: Population firing rate")

# Figure 3: Voltage samples
fig, axes = plt.subplots(2, 1, figsize=(12, 6))
time_range = slice(0, 5000)  # First 500 ms
sample_E = [10, 30, 50, 70]
for nid in sample_E:
    axes[0].plot((np.arange(time_range.stop) * 0.1), signals.V_m[time_range, nid], alpha=0.7, linewidth=0.5)
axes[0].set_ylabel('Voltage (mV)')
axes[0].set_title('Excitatory Neuron Voltage Traces')
axes[0].grid(True, alpha=0.3)
sample_I = [75, 80, 85, 90]
for nid in sample_I:
    axes[1].plot((np.arange(time_range.stop) * 0.1), signals.V_m[time_range, nid], alpha=0.7, linewidth=0.5, color='blue')
axes[1].set_xlabel('Time (ms)')
axes[1].set_ylabel('Voltage (mV)')
axes[1].set_title('Inhibitory Neuron Voltage Traces')
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(figures_dir / "03_voltage_samples.png", dpi=100, bbox_inches='tight')
plt.close()
print("✓ Figure 3: Voltage samples")

# Figure 4: Source summary
fig, ax = plt.subplots(figsize=(12, 4))
if signals.source is not None:
    source_mean = signals.source.mean(axis=1)
    times_ms = np.arange(len(source_mean)) * 0.1
    ax.plot(times_ms, source_mean, color='green', linewidth=1)
    ax.fill_between(times_ms, source_mean.min(), source_mean.max(), alpha=0.3, color='green')
    ax.set_xlabel('Time (ms)')
    ax.set_ylabel('Source Density (a.u.)')
    ax.set_title('Population Source Density')
    ax.grid(True, alpha=0.3)
else:
    ax.text(0.5, 0.5, 'Source data not available', ha='center', va='center')
plt.tight_layout()
plt.savefig(figures_dir / "04_source_summary.png", dpi=100, bbox_inches='tight')
plt.close()
print("✓ Figure 4: Source summary")

# Figure 5: Readout summary
fig, ax = plt.subplots(figsize=(10, 6))
ax.axis('off')
summary_text = f"""100-Neuron E/I Population: Key Statistics

Population Composition:
  • Excitatory neurons: 75 (75%)
  • Inhibitory neurons: 25 (25%)

Firing Rates:
  • Excitatory mean: {rate_E:.2f} Hz
  • Inhibitory mean: {rate_I:.2f} Hz
  • Population mean: {rate_pop:.2f} Hz

Membrane Voltage:
  • Min: {signals.V_m.min():.1f} mV
  • Max: {signals.V_m.max():.1f} mV
  • Mean: {signals.V_m.mean():.1f} mV

Simulation Parameters:
  • Duration: 1000 ms
  • Timestep: 0.1 ms
  • Seed: 42
  • All outputs finite: {bool(jnp.isfinite(signals.V_m).all())}

Scope:
  • Level: Computational scaffold
  • Model: Izhikevich (phenomenological)
  • Field mode: Proxy convolution (no PDE)
  • Physical claims: Not allowed without calibration"""
ax.text(0.1, 0.95, summary_text, transform=ax.transAxes, fontsize=11, 
        verticalalignment='top', family='monospace',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))
plt.savefig(figures_dir / "05_readout_summary.png", dpi=100, bbox_inches='tight')
plt.close()
print("✓ Figure 5: Readout summary")

print(f"\n✓ All figures saved to {figures_dir}")

## 10. Interpretation

This 100-neuron E/I population demonstrates balanced network dynamics. The 75/25 E/I split approximates cortical proportions. Excitatory neurons drive the network with AMPA-mediated input; inhibitory neurons provide feedback inhibition via GABA. Population firing rates typically fall in the 2–25 Hz range for realistic cortical activity. Individual neurons cycle between rest (~-70 mV), depolarization (~-50 mV), spike threshold (~30 mV), and reset (~-65 mV), reflecting Izhikevich phenomenology. Source and field patterns reflect net population activity through spatial averaging and convolution—they are proxy approximations, not full PDE solutions.

## 11. Failure Modes

**Silent Network (< 1 Hz):** Try different seed, switch emitter preset to "tonic_spiking", or adjust connectivity.

**Runaway Network (> 50 Hz):** Use preset with stronger adaptation (e.g., "chattering") or increase inhibitory coupling.

**NaN/Inf in Output:** Reduce timestep to 0.01 ms, use float64 precision, or check Izhikevich parameter bounds.

## 12. Exercises

1. **Vary E/I Ratio:** Change `cfg.cell_types({"E": 0.8, "I": 0.2})` and observe stability.
2. **Try Different Presets:** Use "tonic_spiking", "phasic_spiking", or "fast_spiking" instead of "cortical_eig".
3. **Analyze E vs I Rates:** Compute rates separately by population.
4. **Plot Source-Field Correlation:** Measure coupling between source and LFP.
5. **Multi-Layer Extension:** Define separate columns for L2/3, L4, L5 with inter-layer connections.

## 13. Scope Boundaries

### What This Tutorial Is
✓ A computational scaffold for learning E/I network principles  
✓ A demonstration of configured workflow (not manual kernels)  
✓ A proxy-scale visualization and validation tool  
✓ JAX-based, deterministic, and reproducible  

### What This Tutorial Is NOT
✗ A biophysically realistic neuron model (Izhikevich is phenomenological)  
✗ A validated field solver (LFP/CSD are proxy approximations)  
✗ A tool for direct comparison with real recordings (without calibration)  
✗ A framework for source localization (no inverse problem solve)  
✗ A complete cortical circuit simulator (missing plasticity, neuromodulation)  

### Scope Metadata
```json
{
  "scope_status": "computational_scaffold",
  "calibration_status": "uncalibrated_phenomenological",
  "readout_status": "proxy_scale",
  "field_mode": "proxy_convolution_no_pde",
  "physical_amplitude_claim_allowed": false
}
```